Generating A Preference Dataset With Llama 3.1 70B And Ollama

In [ ]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar（进度条）
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import json
import requests


# 通过本地 Ollama 调用较大的 llama3.1:70b 来「改写」回答，生成 DPO 偏好数据
def query_model(prompt, model="llama3.1:70b", url="http://localhost:11434/api/chat"):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,        # 固定种子，保证可复现
            "temperature": 0,   # 温度 0，确定性输出
        }
    }

    # Send the POST request（流式接收并拼接文本）
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")  # 连通性测试
print(result)

In [ ]:
from pathlib import Path

# 读取第7章指令数据，作为改写的原始素材
json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

In [ ]:
json_data[0]  # 查看第一条结构

In [ ]:
def format_input(entry):
    # 把一条样本拼成 Alpaca 风格指令文本
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    # 【bug修复】原此处有一行独立的 `instruction_text + input_text`(死代码,结果被丢弃),已删除
    return instruction_text + input_text

In [ ]:
import random


# 前 5 条试跑：随机决定把回答改写得更「礼貌」还是更「不礼貌」，用于后续构造偏好对
for entry in json_data[:5]:

    politeness = random.choice(["polite", "impolite"])
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."  # (原文如此，prompt 里有个重复的 return，无伤大雅)
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print(f"\n{politeness} response:")
    print(">>", query_model(prompt))

In [ ]:
import random
from tqdm import tqdm

# 为每条样本生成 chosen/rejected 偏好对：
#   若改写目标是 polite -> 认为「礼貌版」更受偏好(chosen)，原回答为 rejected；
#   若改写目标是 impolite -> 原回答为 chosen，「不礼貌版」为 rejected。
def generate_model_responses(json_data):

    for i, entry in enumerate(tqdm(json_data, desc="Writing entries")):
        politeness = random.choice(["polite", "impolite"])
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        response = query_model(prompt)

        if politeness == "polite":
            json_data[i]["chosen"] = response          # 礼貌改写版作为偏好答案
            json_data[i]["rejected"] = entry["output"]
        else:
            json_data[i]["rejected"] = response        # 不礼貌改写版作为被拒答案
            json_data[i]["chosen"] = entry["output"]

In [ ]:
generate_model_responses(json_data)  # 对全部数据生成 chosen/rejected 字段

In [ ]:
# 保存带偏好标注(chosen/rejected)的数据，供后续 DPO 训练使用
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)